
# E-Ambulance — EDA trước khi tích hợp OSMnx

Notebook này dùng để **kiểm tra chất lượng dữ liệu, phân tích phân phối và chuẩn bị các điểm địa lý** trước khi thay thế khoảng cách/thời gian surrogate bằng mạng đường OpenStreetMap/OSMnx.

## Mục tiêu

1. Kiểm tra schema và kích thước các bảng.
2. Kiểm tra `latitude/longitude`, missing values và duplicate coordinates.
3. Phân tích:
   - số incident theo scenario;
   - priority;
   - specialty;
   - thời điểm phát sinh incident;
   - SOC ban đầu của fleet;
   - trạng thái capacity bệnh viện.
4. Trực quan hóa phân bố tọa độ incident/fleet/hospital/charger.
5. Sinh:
   - `osmnx_points.csv`;
   - `osmnx_bbox.json`;
   - các file QA cho bước routing.
6. Chuẩn bị logic tích hợp OSMnx nhưng **chưa tải road graph trong notebook EDA này**.

> **Lưu ý khoa học:** dataset hiện tại là synthetic benchmark. Hospital capacity là trạng thái mô phỏng, không phải số giường/công suất tiếp nhận thực tế của bệnh viện TP.HCM.


In [ ]:

from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

# Tự tìm project root để notebook chạy được khi mở từ project root hoặc thư mục notebooks/
cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, *cwd.parents]

ROOT = None
for p in candidates:
    if (p / "data" / "sample").exists():
        ROOT = p
        break

if ROOT is None:
    raise FileNotFoundError(
        "Không tìm thấy data/sample. Hãy mở notebook từ thư mục eambulance_dataset_v2_rebuilt "
        "hoặc sửa ROOT thủ công."
    )

DATA = ROOT / "data" / "sample"
OUT = ROOT / "eda_pre_osmnx"
OUT.mkdir(exist_ok=True)

print("ROOT =", ROOT)
print("DATA =", DATA)
print("OUT  =", OUT)


## 1. Load dữ liệu

In [ ]:

inc = pd.read_csv(DATA / "incidents.csv.gz")
fleet = pd.read_csv(DATA / "fleet_initial.csv.gz")
hosp = pd.read_csv(DATA / "hospitals.csv")
chg = pd.read_csv(DATA / "charging_stations.csv")
cap = pd.read_csv(DATA / "hospital_capacity_timeseries.csv.gz")
reps = pd.read_csv(DATA / "replications.csv.gz")
scenarios = pd.read_csv(DATA / "scenarios.csv.gz")

datasets = {
    "incidents": inc,
    "fleet_initial": fleet,
    "hospitals": hosp,
    "charging_stations": chg,
    "hospital_capacity_timeseries": cap,
    "replications": reps,
    "scenarios": scenarios,
}

summary = pd.DataFrame({
    "dataset": list(datasets),
    "rows": [len(df) for df in datasets.values()],
    "columns": [len(df.columns) for df in datasets.values()],
})

summary


### Kiểm tra schema nhanh

In [ ]:

for name, df in datasets.items():
    print(f"\n[{name}]")
    print(", ".join(df.columns))


## 2. Kiểm tra missing values

In [ ]:

missing_summary = []

for name, df in datasets.items():
    n_missing = int(df.isna().sum().sum())
    cols_missing = int((df.isna().sum() > 0).sum())
    missing_summary.append({
        "dataset": name,
        "rows": len(df),
        "total_missing_cells": n_missing,
        "columns_with_missing": cols_missing,
    })

pd.DataFrame(missing_summary)


## 3. Coordinate QA

In [ ]:

def coordinate_qa(name, df, lat_col, lon_col):
    return {
        "dataset": name,
        "rows": len(df),
        "missing_lat": int(df[lat_col].isna().sum()),
        "missing_lon": int(df[lon_col].isna().sum()),
        "invalid_lat": int(((df[lat_col] < -90) | (df[lat_col] > 90)).sum()),
        "invalid_lon": int(((df[lon_col] < -180) | (df[lon_col] > 180)).sum()),
        "lat_min": float(df[lat_col].min()),
        "lat_max": float(df[lat_col].max()),
        "lon_min": float(df[lon_col].min()),
        "lon_max": float(df[lon_col].max()),
        "unique_coords": int(df[[lat_col, lon_col]].drop_duplicates().shape[0]),
        "duplicate_coord_rows": int(df.duplicated([lat_col, lon_col]).sum()),
    }

coord_qa_df = pd.DataFrame([
    coordinate_qa("incidents", inc, "incident_lat", "incident_lon"),
    coordinate_qa("fleet_initial", fleet, "latitude", "longitude"),
    coordinate_qa("hospitals", hosp, "latitude", "longitude"),
    coordinate_qa("charging_stations", chg, "latitude", "longitude"),
])

coord_qa_df


In [ ]:

coord_qa_df.to_csv(OUT / "coordinate_qa.csv", index=False)
print("Saved:", OUT / "coordinate_qa.csv")


## 4. Spatial overview — tọa độ trước khi snap vào OSM nodes

In [ ]:

fig, ax = plt.subplots(figsize=(9, 7))

ax.scatter(inc["incident_lon"], inc["incident_lat"], s=8, alpha=0.18, label="Incidents")
ax.scatter(fleet["longitude"], fleet["latitude"], s=18, alpha=0.45, label="Fleet initial")
ax.scatter(hosp["longitude"], hosp["latitude"], s=60, marker="^", label="Hospitals")
ax.scatter(chg["longitude"], chg["latitude"], s=60, marker="s", label="Chargers")

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Phân bố tọa độ đầu vào trước OSMnx")
ax.legend()
ax.grid(alpha=0.25)

plt.show()



### Nhận xét cần kiểm soát

- Incident phân bố trên vùng rộng hơn fleet/hospital/charger.
- Khi dùng OSMnx, tất cả điểm phải được **snap vào node/edge của road graph**.
- Cần lưu `snap_distance_m`; các điểm có snap distance quá lớn phải được review thay vì tự động chấp nhận.


## 5. Incident theo scenario

In [ ]:

scenario_counts = inc["scenario_id"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(scenario_counts.index, scenario_counts.values)
ax.set_xlabel("Scenario")
ax.set_ylabel("Number of incidents")
ax.set_title("Số incident theo scenario")
ax.grid(axis="y", alpha=0.25)
plt.show()

scenario_counts.to_frame("incident_count")


## 6. Priority distribution

In [ ]:

priority_counts = (
    inc.groupby(["scenario_id", "priority"])
       .size()
       .unstack(fill_value=0)
       .sort_index()
)

priority_counts


In [ ]:

priority_counts.plot(kind="bar", figsize=(9, 5))
plt.xlabel("Scenario")
plt.ylabel("Incident count")
plt.title("Priority distribution theo scenario")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.25)
plt.show()


## 7. Specialty distribution và hospital coverage

In [ ]:

specialty_counts = inc["required_specialty"].value_counts().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(specialty_counts.index, specialty_counts.values)
ax.set_xlabel("Required specialty")
ax.set_ylabel("Incident count")
ax.set_title("Required specialty distribution")
plt.xticks(rotation=35, ha="right")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

specialty_counts.to_frame("incident_count")


In [ ]:

support = {}
for _, row in hosp.iterrows():
    specialties = [s.strip() for s in str(row["supported_specialties"]).split("|") if s.strip()]
    for s in specialties:
        support.setdefault(s, []).append(row["hospital_id"])

coverage_rows = []
for specialty, count in specialty_counts.items():
    hs = support.get(specialty, [])
    coverage_rows.append({
        "required_specialty": specialty,
        "incident_count": int(count),
        "supporting_hospital_count": len(hs),
        "supporting_hospital_ids": "|".join(hs),
        "coverage_flag": "OK" if hs else "NO_HOSPITAL_SUPPORT",
    })

specialty_coverage = pd.DataFrame(coverage_rows).sort_values(
    ["supporting_hospital_count", "required_specialty"]
)

specialty_coverage.to_csv(OUT / "specialty_coverage.csv", index=False)
specialty_coverage


In [ ]:

fig, ax = plt.subplots(figsize=(9, 5))
tmp = specialty_coverage.sort_values("supporting_hospital_count")
ax.bar(tmp["required_specialty"], tmp["supporting_hospital_count"])
ax.set_xlabel("Required specialty")
ax.set_ylabel("Supporting hospitals")
ax.set_title("Số bệnh viện hỗ trợ theo specialty")
plt.xticks(rotation=35, ha="right")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()



### Điểm cần theo dõi

Nếu một specialty chỉ có 1 bệnh viện hỗ trợ, network travel time có thể làm **hospital feasibility trở thành bottleneck**.  
Đây không nhất thiết là lỗi dữ liệu, nhưng cần được đánh dấu trong sensitivity analysis.


## 8. Phân tích thời điểm incident

In [ ]:

inc["timestamp"] = pd.to_datetime(inc["timestamp"], utc=True)
inc["hour"] = inc["timestamp"].dt.hour + inc["timestamp"].dt.minute / 60

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(inc["hour"], bins=np.arange(0, 25, 1))
ax.set_xlabel("Hour of day")
ax.set_ylabel("Incident count")
ax.set_title("Phân bố thời điểm phát sinh incident")
ax.set_xticks(range(0, 25, 2))
ax.grid(axis="y", alpha=0.25)
plt.show()


In [ ]:

hour_by_scenario = (
    inc.assign(hour_int=inc["timestamp"].dt.hour)
       .groupby(["scenario_id", "hour_int"])
       .size()
       .unstack(fill_value=0)
)

fig, ax = plt.subplots(figsize=(10, 5))
for scenario_id, row in hour_by_scenario.iterrows():
    ax.plot(row.index, row.values, marker="o", label=scenario_id)

ax.set_xlabel("Hour")
ax.set_ylabel("Incident count")
ax.set_title("Incident intensity theo giờ và scenario")
ax.legend()
ax.grid(alpha=0.25)
plt.show()


## 9. Fleet initial state và SOC

In [ ]:

fleet_state_counts = fleet["vehicle_state_init"].value_counts()

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(fleet_state_counts.index, fleet_state_counts.values)
ax.set_xlabel("Initial vehicle state")
ax.set_ylabel("Count")
ax.set_title("Fleet initial state distribution")
ax.grid(axis="y", alpha=0.25)
plt.show()

fleet_state_counts.to_frame("count")


In [ ]:

fleet["soc_init_pct"].describe()


In [ ]:

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(fleet["soc_init_pct"], bins=15)
ax.axvline(fleet["reserve_soc_pct"].median(), linestyle="--", label="Median reserve SOC")
ax.set_xlabel("Initial SOC (%)")
ax.set_ylabel("Vehicle-state rows")
ax.set_title("Initial SOC distribution")
ax.legend()
ax.grid(axis="y", alpha=0.25)
plt.show()



### Cảnh báo từ QA trước

Nếu SOC sensitivity gần như phẳng trong khoảng reserve SOC 10–30%, cần tăng `energy_stress` trước khi dùng B4/B6 để SOC thực sự làm thay đổi feasible set.


## 10. Hospital capacity state theo scenario

In [ ]:

capacity_state = (
    cap.groupby(["scenario_id", "capacity_state"])
       .size()
       .unstack(fill_value=0)
)

capacity_state


In [ ]:

capacity_state.plot(kind="bar", figsize=(9, 5))
plt.xlabel("Scenario")
plt.ylabel("Time-bin rows")
plt.title("Hospital capacity state theo scenario")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.25)
plt.show()


## 11. Tạo bảng điểm dùng cho OSMnx

In [ ]:

frames = [
    pd.DataFrame({
        "point_type": "incident",
        "entity_id": inc["incident_id"],
        "scenario_id": inc["scenario_id"],
        "replication_id": inc["replication_id"],
        "latitude": inc["incident_lat"],
        "longitude": inc["incident_lon"],
    }),
    pd.DataFrame({
        "point_type": "fleet_initial",
        "entity_id": fleet["vehicle_id"],
        "scenario_id": fleet["scenario_id"],
        "replication_id": fleet["replication_id"],
        "latitude": fleet["latitude"],
        "longitude": fleet["longitude"],
    }),
    pd.DataFrame({
        "point_type": "hospital",
        "entity_id": hosp["hospital_id"],
        "scenario_id": "",
        "replication_id": "",
        "latitude": hosp["latitude"],
        "longitude": hosp["longitude"],
    }),
    pd.DataFrame({
        "point_type": "charger",
        "entity_id": chg["charger_id"],
        "scenario_id": "",
        "replication_id": "",
        "latitude": chg["latitude"],
        "longitude": chg["longitude"],
    }),
]

osmnx_points = pd.concat(frames, ignore_index=True)
osmnx_points.to_csv(OUT / "osmnx_points.csv", index=False)

print("Rows:", len(osmnx_points))
print("Unique coordinates:", osmnx_points[["latitude", "longitude"]].drop_duplicates().shape[0])
osmnx_points.head()


## 12. Bounding box cho bước tải graph

In [ ]:

lat_min = osmnx_points["latitude"].min()
lat_max = osmnx_points["latitude"].max()
lon_min = osmnx_points["longitude"].min()
lon_max = osmnx_points["longitude"].max()

lat_mid = (lat_min + lat_max) / 2
north_south_km = (lat_max - lat_min) * 111.32
east_west_km = (lon_max - lon_min) * 111.32 * math.cos(math.radians(lat_mid))

buffer_km = 2.0
lat_buffer = buffer_km / 111.32
lon_buffer = buffer_km / (111.32 * math.cos(math.radians(lat_mid)))

bbox = {
    "raw": {
        "south": float(lat_min),
        "north": float(lat_max),
        "west": float(lon_min),
        "east": float(lon_max),
        "approx_north_south_km": round(float(north_south_km), 3),
        "approx_east_west_km": round(float(east_west_km), 3),
    },
    "suggested_graph_download_bbox_with_2km_margin": {
        "south": round(float(lat_min - lat_buffer), 6),
        "north": round(float(lat_max + lat_buffer), 6),
        "west": round(float(lon_min - lon_buffer), 6),
        "east": round(float(lon_max + lon_buffer), 6),
        "note": "2 km chỉ là download margin thực dụng; không phải tham số khoa học cố định."
    }
}

(OUT / "osmnx_bbox.json").write_text(json.dumps(bbox, indent=2), encoding="utf-8")
bbox


In [ ]:

fig, ax = plt.subplots(figsize=(9, 7))

ax.scatter(inc["incident_lon"], inc["incident_lat"], s=7, alpha=0.18)

raw = bbox["raw"]
buffered = bbox["suggested_graph_download_bbox_with_2km_margin"]

# raw bbox
ax.plot(
    [raw["west"], raw["east"], raw["east"], raw["west"], raw["west"]],
    [raw["south"], raw["south"], raw["north"], raw["north"], raw["south"]],
    label="Raw bbox"
)

# buffered bbox
ax.plot(
    [buffered["west"], buffered["east"], buffered["east"], buffered["west"], buffered["west"]],
    [buffered["south"], buffered["south"], buffered["north"], buffered["north"], buffered["south"]],
    linestyle="--",
    label="BBox + 2 km margin"
)

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Study bounding box cho OSMnx")
ax.legend()
ax.grid(alpha=0.25)
plt.show()



## 13. Thiết kế OD matrix trước OSMnx

Không nên gọi shortest path trực tiếp cho toàn bộ 540.000 candidate rows.

Nên tách thành các leg:

### A. Fleet-origin → Incident

Trạng thái ban đầu dùng cho reference/smoke-test. Khi rolling DES chạy, origin của xe sẽ thay đổi theo policy nên OD này phải được cập nhật/cached theo state.

### B. Incident → Hospital

Tối đa:

\[
9000 	imes 6 = 54\,000
\]

cặp trước feasibility filtering.

### C. Hospital → Charger

Chỉ:

\[
6 	imes 4 = 24
\]

cặp.

Sau đó candidate `Ambulance × Incident × Hospital` được ghép từ các leg đã tính/cached.


In [ ]:

od_size = pd.DataFrame([
    {"leg": "incident_to_hospital", "origin_count": inc["incident_id"].nunique(),
     "destination_count": hosp["hospital_id"].nunique(),
     "max_pairs": inc["incident_id"].nunique() * hosp["hospital_id"].nunique()},
    {"leg": "hospital_to_charger", "origin_count": hosp["hospital_id"].nunique(),
     "destination_count": chg["charger_id"].nunique(),
     "max_pairs": hosp["hospital_id"].nunique() * chg["charger_id"].nunique()},
])

od_size



# 14. Cell chuẩn bị cho OSMnx — **chưa chạy tự động**

Sau khi EDA đạt yêu cầu, cài:

```bash
pip install osmnx
```

Sau đó dùng cell dưới đây trong notebook tiếp theo để:

1. tải graph một lần;
2. thêm `speed_kph`;
3. thêm `travel_time`;
4. snap points vào node;
5. lưu `study_graph.graphml`;
6. lưu `point_node_map.csv`.

> Không nên chạy request mạng lặp lại cho từng incident/candidate.


In [ ]:

# CHỈ CHẠY SAU KHI ĐÃ `pip install osmnx`
#
# import osmnx as ox
#
# b = bbox["suggested_graph_download_bbox_with_2km_margin"]
#
# # Kiểm tra API của phiên bản OSMnx đang cài trước khi chạy.
# print("OSMnx version:", ox.__version__)
#
# # Ví dụ logic tổng quát:
# # G = ox.graph_from_bbox(
# #     north=b["north"],
# #     south=b["south"],
# #     east=b["east"],
# #     west=b["west"],
# #     network_type="drive",
# #     simplify=True,
# # )
# #
# # G = ox.routing.add_edge_speeds(G)
# # G = ox.routing.add_edge_travel_times(G)
# #
# # node_ids = ox.distance.nearest_nodes(
# #     G,
# #     X=osmnx_points["longitude"].to_numpy(),
# #     Y=osmnx_points["latitude"].to_numpy(),
# # )
# #
# # point_node_map = osmnx_points.copy()
# # point_node_map["osmid"] = node_ids
# #
# # ox.save_graphml(G, ROOT / "data" / "osmnx" / "study_graph.graphml")
# # point_node_map.to_csv(ROOT / "data" / "osmnx" / "point_node_map.csv", index=False)
#
# print("Cell này là template; chưa tải road graph trong notebook EDA.")


## 15. Xuất summary cho bước tiếp theo

In [ ]:

eda_summary = {
    "rows": {
        "incidents": int(len(inc)),
        "fleet_initial": int(len(fleet)),
        "hospitals": int(len(hosp)),
        "charging_stations": int(len(chg)),
        "hospital_capacity_timeseries": int(len(cap)),
        "osmnx_points_total": int(len(osmnx_points)),
        "osmnx_unique_coordinates": int(osmnx_points[["latitude", "longitude"]].drop_duplicates().shape[0]),
    },
    "bbox": bbox,
    "priority_counts": {str(k): int(v) for k, v in inc["priority"].value_counts().to_dict().items()},
    "specialty_counts": {str(k): int(v) for k, v in inc["required_specialty"].value_counts().to_dict().items()},
    "fleet_state_counts": {str(k): int(v) for k, v in fleet["vehicle_state_init"].value_counts().to_dict().items()},
    "soc_init_pct": {
        "min": float(fleet["soc_init_pct"].min()),
        "mean": float(fleet["soc_init_pct"].mean()),
        "median": float(fleet["soc_init_pct"].median()),
        "max": float(fleet["soc_init_pct"].max()),
    },
}

(OUT / "eda_summary.json").write_text(
    json.dumps(eda_summary, indent=2),
    encoding="utf-8"
)

print("EDA outputs:")
for p in sorted(OUT.glob("*")):
    print("-", p.name)



# 16. Gate trước khi chuyển sang OSMnx

Chỉ chuyển sang network build khi các điều kiện sau đều đạt:

- [ ] Không có missing lat/lon.
- [ ] Không có coordinate ngoài miền hợp lệ.
- [ ] Tất cả specialty được yêu cầu có ít nhất một bệnh viện hỗ trợ.
- [ ] BBox không có outlier vô lý.
- [ ] Incident intensity theo giờ/scenario phù hợp với generator design.
- [ ] Initial SOC distribution đủ hợp lý cho thí nghiệm.
- [ ] Capacity states đúng theo scenario assumptions.
- [ ] Đã lưu `osmnx_points.csv`.
- [ ] Đã lưu `osmnx_bbox.json`.
- [ ] Sau khi snap OSM, sẽ kiểm tra `snap_distance_m`.

## Bước tiếp theo

Tạo notebook `02_Build_OSMnx_Network.ipynb` để tải graph, snap point, tính `network_distance_m` và `freeflow_travel_time_s`, sau đó build OD cache.
